# V7.6 — Symmetric-loss V7 retrains on Kaggle GPU

Trains three symmetric-objective variants of the V7 base model using the existing V7 ABT + features:

* **Tweedie** (`objective='tweedie'`, variance_power 1.3) — count-like, handles zero-inflation natively.
* **MAE** (`objective='regression_l1'`) — L1 loss, directly aligned with WAPE.
* **Huber** (`objective='huber'`, alpha=0.9) — robust L2/L1 hybrid.

Each variant writes `preds_v7sym_{name}_{val,test}.csv` to `/kaggle/working` so they can be pulled locally and fed into the V7.6 LAD stacker pool (`scripts/v75_lad_stack.py`) alongside V4..V7.2.

Runtime budget: ~3 GPU-minutes per variant on Kaggle T4 (vs 30+ min stalled on laptop CPU with OpenMP issues). No Optuna — the V7.3 → V7.5 story showed that the stacker gets almost all the lift, so we just want well-trained symmetric base models.

In [ ]:
import shutil, subprocess, sys
if shutil.which('nvidia-smi'):
    print(subprocess.check_output(['nvidia-smi', '-L']).decode())
else:
    print('CPU-only kernel (fine — LightGBM will still run fast here without macOS OMP issues)')
print(sys.version)

In [ ]:
!pip install -q lightgbm==4.6.0 joblib pandas pyarrow scikit-learn
import os, shutil, sys
INPUT_DIR = '/kaggle/input'
REPO_DIR = '/kaggle/working/repo'
os.makedirs(REPO_DIR, exist_ok=True)
print('kaggle/input contents:', os.listdir(INPUT_DIR))

candidates = [d for d in os.listdir(INPUT_DIR) if os.path.isdir(f'{INPUT_DIR}/{d}')]
assert candidates, 'no dataset attached'
DATASET = next((d for d in candidates if 'bpm' in d.lower()), candidates[0])
DS_DIR = f'{INPUT_DIR}/{DATASET}'
print('using dataset', DS_DIR)

def _locate(name):
    if os.path.isdir(f'{DS_DIR}/{name}'): return f'{DS_DIR}/{name}'
    for root, dirs, _ in os.walk(DS_DIR):
        if name in dirs: return os.path.join(root, name)
    return None

for sub in ('src', 'scripts'):
    loc = _locate(sub)
    assert loc, f'missing {sub}/'
    shutil.copytree(loc, f'{REPO_DIR}/{sub}', dirs_exist_ok=True)

os.makedirs(f'{REPO_DIR}/output', exist_ok=True)
def _find_file(name):
    for root, _, files in os.walk(DS_DIR):
        if name in files: return os.path.join(root, name)
    return None

for fn in ('abt_v7_cached.parquet', 'abt_v6_cached.parquet'):
    found = _find_file(fn)
    if found:
        shutil.copy(found, f'{REPO_DIR}/output/{fn}'); print('copied', fn)
    else:
        print('missing', fn)

os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print('ready')

In [ ]:
import numpy as np, pandas as pd, time, json
from src.evaluation import split_train_val_test
from src.model_v2 import (
    TwoStageForecaster, encode_categoricals,
    filter_active_pairs, get_feature_columns_v2,
)

abt = pd.read_parquet('output/abt_v7_cached.parquet').pipe(encode_categoricals)
feats = [c for c in get_feature_columns_v2(abt) if c != 'target_qty_imputed']
print(f'ABT: {len(abt)} rows  |  {len(feats)} features')

df_train, df_val, df_test = split_train_val_test(abt)
active = filter_active_pairs(df_train)
keys = active[['Партнер', 'Артикул']].drop_duplicates()
df_val = df_val.merge(keys, on=['Партнер', 'Артикул'], how='inner')
df_test = df_test.merge(keys, on=['Партнер', 'Артикул'], how='inner')
print(f'train={len(active)}  val={len(df_val)}  test={len(df_test)}')

In [ ]:
# Common training parameters.  We keep the V7 classifier stage identical
# across all symmetric variants and only change the regression objective.
KEY = ['Период', 'Партнер', 'Артикул']

COMMON_CLF = {
    'num_leaves': 127,
    'learning_rate': 0.05,
    'min_child_samples': 30,
    'verbose': -1,
}

COMMON_REG = {
    'num_leaves': 127,
    'learning_rate': 0.05,
    'min_child_samples': 20,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 5,
    'verbose': -1,
}

OBJECTIVES = {
    'tweedie':  {'objective': 'tweedie', 'tweedie_variance_power': 1.3},
    'tweedie15': {'objective': 'tweedie', 'tweedie_variance_power': 1.5},
    'mae':      {'objective': 'regression_l1'},
    'huber':    {'objective': 'huber', 'alpha': 0.9},
    'l2':       {'objective': 'regression'},
}

def save_preds(df, preds, path):
    out = df[KEY + ['target_qty']].copy()
    out['prediction'] = np.clip(preds, 0, None)
    out.to_csv(path, index=False)

def metric(y, p):
    wape = float(np.abs(y - p).sum() / max(float(y.sum()), 1.0))
    bias = float((p.sum() - y.sum()) / max(float(y.sum()), 1.0) * 100.0)
    return wape, bias

metrics_rows = []

for name, reg_extra in OBJECTIVES.items():
    print(f'\n======= training v7sym_{name} =======')
    reg_params = {**COMMON_REG, **reg_extra}
    model = TwoStageForecaster(
        clf_params=COMMON_CLF,
        reg_params=reg_params,
        reg_objective='',
        target_col='target_qty_imputed',
    )
    t0 = time.time()
    model.fit(active, df_val, feats, num_boost_round=1200, early_stopping=50)
    dt = time.time() - t0
    p_val = model.predict(df_val)
    p_test = model.predict(df_test)
    save_preds(df_val, p_val, f'/kaggle/working/preds_v7sym_{name}_val.csv')
    save_preds(df_test, p_test, f'/kaggle/working/preds_v7sym_{name}_test.csv')
    y_val = df_val['target_qty'].to_numpy()
    y_test = df_test['target_qty'].to_numpy()
    vw, vb = metric(y_val, p_val)
    tw, tb = metric(y_test, p_test)
    print(f'  fit in {dt:.1f}s  |  val WAPE={vw:.4f} bias%={vb:+.2f}  '
          f'|  test WAPE={tw:.4f} bias%={tb:+.2f}')
    metrics_rows.append({'name': name, 'fit_sec': round(dt, 1),
                         'val_WAPE': vw, 'val_bias_pct': vb,
                         'test_WAPE': tw, 'test_bias_pct': tb})

pd.DataFrame(metrics_rows).to_csv('/kaggle/working/v7sym_metrics.csv', index=False)
print('\nall variants trained; preds saved to /kaggle/working')
print(pd.DataFrame(metrics_rows).to_string(index=False))

In [ ]:
# Bundle outputs into a single zip for easy pull.
import os, zipfile
with zipfile.ZipFile('/kaggle/working/v7sym_bundle.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir('/kaggle/working'):
        if f.startswith('preds_v7sym_') or f == 'v7sym_metrics.csv':
            z.write(f'/kaggle/working/{f}', f)
            print('zipped', f)
print('\nbundle at /kaggle/working/v7sym_bundle.zip')